<a href="https://colab.research.google.com/github/sxynix32/gdp-dashboard/blob/main/HealthMate_%F0%9F%A7%A0_%E2%80%93_Feels_like_a_companion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install streamlit faiss-cpu sentence-transformers langchain-community pyngrok pypdf groq langchain-groq

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 92.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 82.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 116.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.3/302.3 kB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.5/127.5 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 143.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 149.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 109.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 69.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664

In [2]:
import os
os.environ["GROQ_API_KEY"] ="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"

In [3]:
app_code = '''
import os
import streamlit as st
from langchain.vectorstores import FAISS
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.llms import HuggingFaceHub
from langchain.chains import RetrievalQA
from langchain. prompts import PromptTemplate
from langchain.chains.question_answering import load_qa_chain
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.docstore.document import Document
from langchain_groq import ChatGroq




def load_medical_docs(path):
    loader = PyPDFLoader(path)
    docs = loader.load()
    print(f'{len(docs)}')
    splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
    return splitter.split_documents(docs)



def embed_documents(docs):
    embed_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    return FAISS.from_documents(docs, embed_model)



def build_qa_system(faiss_index):
    retriever = faiss_index.as_retriever(search_type="similarity", k=4)
    llm = ChatGroq(
       api_key=os.getenv("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"),
       model_name= "llama3-8b-8192"
       )
    qa_chain = RetrievalQA.from_chain_type(llm=llm, retriever=retriever)
    return qa_chain


def main():
    st.set_page_config(page_title="Medical Chatbot (LLaMA 3 + RAG)", layout="centered")
    st.title("HealthMate 🧠 – Feels like a companion")

    if "qa_chain" not in st.session_state:
        with st.spinner("Embedding medical documents..."):
           docs = load_medical_docs("/content/Oxford Handbook of Clinical Medicine 10th 2017 Edition_SamanSarKo.pdf")
           faiss_index = embed_documents(docs)
           st.session_state.qa_chain = build_qa_system(faiss_index)



    query = st.text_input("Ask a medical question:")
    if query:
        with st.spinner ("Generating answer..."):
            result = st.session_state.qa_chain.run(query)
            st.markdown(f"**Answer:** {result}")



if __name__ == "__main__":
    main()
'''
with open ("Health_Mate.py", "w") as f:
    f.write(app_code)

In [5]:
from pyngrok import ngrok
!ngrok config add-authtoken 2wXshtUgYiKALhwAN5FatTISg4f_7rPbCs4HgkzQSH51Z7iZr
public_url = ngrok.connect(8501)
print(f"stearmlit app is live at:{public_url}")

!streamlit run Health_Mate.py &>/content/logs.logs.txt &

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml
stearmlit app is live at:NgrokTunnel: "https://510a-34-16-252-90.ngrok-free.app" -> "http://localhost:8501"
